<a href="https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Data contract

**Unit of analysis**

One row represents one content page for one client on one reporting date.

**Time window**

This notebook focuses on the March 2026 snapshot (`month = '2026-03'`), following the recommendation to use a mid-panel month instead of the final month.

This contract supports the **Refresh / Content Opportunity Scoring** lane. The final modeling dataset will later aggregate daily observations into page-level features, but this notebook verifies the source data at its original grain.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [5]:
march = f"""
SELECT *
FROM read_parquet(
'{DATASET}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
"""
df = con.sql(march).df()
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [6]:
print(df.head())

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...     

In [7]:
# Query 1 — Verify the grain
grain_check = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM df
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()

print("Duplicate rows:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows: 0


In [8]:
# Query 2 — Count and date window
summary = con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM df
""").df()

summary

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# Query 3 — Availability
availability = con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM df
WHERE gsc_data_available IS TRUE
""").df()

availability

,available_rows
0,3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Planned fields

### Features

- `gsc_impressions`
- `gsc_clicks`
- `gsc_sum_position`
- `client_has_gsc`
- `gsc_data_available`

These fields are available before the editorial decision and can be used to construct page-level features later in the project.

### Label / Proxy

The project ultimately aims to rank pages for editorial review.

Because the warehouse does not contain the true outcome of future editorial interventions, a proxy label will be used during development.

### Context

- `report_date`
- `month`

These variables define the observation window but are not intended as predictive features.

### Excluded

- `client_hash_id`
- `content_hash_id`

These identifiers uniquely identify observations but should not be used as predictive features.

Future information or any label-derived field will also be excluded to avoid data leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This warehouse contains observational data only.

It can support prioritizing pages for editorial review, but it cannot determine whether refreshing a page actually caused future improvements.

The dataset is also an unbalanced panel, meaning that different clients have different history lengths and data availability. Additionally, some rows do not contain Google Search Console data, so availability checks are required before modeling.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.